# SD1.5 Prompt-Mismatched In-Range Recovery Results

This notebook analyzes the original unweighted reconstruction pipeline across
the four empirical Christoffel sampling laws, uniform MCS, and pure
inverse-square sampling. All six distributions use the original seven sampling
ratios and five trials. PSNR, SSIM, LPIPS, and per-pixel MAE are loaded and
plotted through the shared analysis code. Outputs remain under
the corresponding experiment's `results/unweighted/<scenario>/sunset/figures/`
directory.


In [ ]:
from pathlib import Path
import importlib
import sys

from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
for search_root in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    helper_dir = search_root / 'analyze_results'
    helper_path = helper_dir / 'sd15_recovery_analysis.py'
    if helper_path.exists():
        if str(helper_dir) not in sys.path:
            sys.path.insert(0, str(helper_dir))
        break
    for child in search_root.iterdir():
        if not child.is_dir():
            continue
        helper_dir = child / 'analyze_results'
        helper_path = helper_dir / 'sd15_recovery_analysis.py'
        if helper_path.exists():
            if str(helper_dir) not in sys.path:
                sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise FileNotFoundError('Could not find sd15_recovery_analysis.py from the notebook cwd.')

import sd15_recovery_analysis as recovery
recovery = importlib.reload(recovery)

SD15_ROOT = recovery.find_sd15_root(NOTEBOOK_DIR)
UNWEIGHTED_BASE_TAG = 'unweighted/prompt_mismatched/sunset'
SAMPLING_METHODS = list(recovery.UNWEIGHTED_MAIN_SAMPLING_METHODS)
ALLOWED_SAMPLING_PERC = set(recovery.DEFAULT_ALLOWED_SAMPLING_PERCENTAGES)
OUTPUT_ROOT = SD15_ROOT / 'results'

# LPIPS is calculated by the shared analysis pipeline and saved incrementally.
LPIPS_TABLE = recovery.ensure_lpips_metrics(
    SD15_ROOT,
    result_namespace='unweighted',
    artifact_roots=[SD15_ROOT / 'results' / UNWEIGHTED_BASE_TAG],
    device='cpu',
)
analysis, COMPLETION_TABLE = recovery.load_unweighted_main_analysis(
    SD15_ROOT,
    base_tag=UNWEIGHTED_BASE_TAG,
    output_root=OUTPUT_ROOT,
    include_partial=True,
)
ROWS = analysis.rows
MEAN_TABLE = analysis.mean_table
ACTIVE_TAG = analysis.active_tag
LOADED_TAGS = analysis.loaded_tags
OUTPUT_DIR = analysis.output_dir / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPLETION_PATH = OUTPUT_DIR / 'unweighted_main_completion.csv'
COMPLETION_TABLE.to_csv(COMPLETION_PATH, index=False)

print(f'Active tag: {ACTIVE_TAG}')
print(f'Loaded source tags: {LOADED_TAGS}')
print(f'Loaded {len(ROWS)} / 840 expected run rows.')
lpips_count = int(ROWS['lpips'].notna().sum()) if 'lpips' in ROWS else 0
print(f'Loaded {lpips_count} LPIPS values.')
display(
    COMPLETION_TABLE.groupby(
        ['sampling_method', 'sampling_condition'],
        as_index=False,
    )[['observed', 'expected', 'left']].sum()
)
display(MEAN_TABLE)
if MEAN_TABLE.empty:
    print('No recovery rows found yet. Run the suite first, then rerun this notebook.')


## Metric Curves

This cell calls the shared recovery plotting helpers to export metric curves for `psnr_db`, `ssim`, `lpips`, and `pixel_mae` plus a combined PSNR/SSIM panel for each diffusion/sampling method that has rows. Each subplot fixes a sampling prior, the colored lines compare recovery prompts, and the x-axis is the sampling ratio `m/n` on a log scale.

Curves show the mean over repeats. The shaded region is a 95% normal-approximation confidence interval, computed as mean +/- 1.96 SEM in the plotted metric units. The black dashed reference is the zero-filled inverse FFT baseline when those metrics are present.

In [ ]:
METRIC_OUTPUTS = recovery.export_metric_figures(
    ROWS,
    OUTPUT_DIR,
    combine_sampling_methods=True,
    show=True,
)
METRIC_OUTPUTS


## Recovery Grid

This cell builds the image grids used to inspect reconstruction quality directly. For each sampling prior, it selects one target item and one sampling ratio, then shows the ground truth, the zero-filled inverse FFT baseline, and the best available reconstruction for each recovery prompt.

The "best" reconstruction in each tile is selected from the loaded rows by minimum LPIPS, with maximum PSNR as the tie-breaker, so the grid is a compact visual counterpart to the metric curves. The cell saves one PDF per sampling prior in `OUTPUT_DIR` and displays the figures inline.

In [ ]:
IMAGE_SAMPLING_PERC = 0.00125

GRID_OUTPUTS = recovery.export_recovery_grids(
    ROWS,
    SD15_ROOT,
    OUTPUT_DIR,
    sampling_method=None,
    sampling_percentage=IMAGE_SAMPLING_PERC,
    show=True,
)
GRID_OUTPUTS
